# CS 8803 Soccer-Twos Project Runner

This is the complete local project notebook. It assumes your `soccertwos` environment already exists and `requirements.txt` is installed. The notebook is intentionally split into small sections so you can understand what is happening before launching long training runs.

## Mental Model

The project has four phases:

1. **Environment sanity check:** verify Soccer-Twos launches and understand observation/action/reward.
2. **Training:** run PPO baseline, PPO reward-shaped, and PPO curriculum agents with Ray/RLlib.
3. **Monitoring:** watch live TensorBoard logs and inspect `progress.csv` tables/plots.
4. **Submission:** export compact `AgentInterface` packages, evaluate them, and zip them for grading.

## 0. Project Setup

In [ ]:
from pathlib import Path
import sys

def _add_project_root_to_path():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "soccer-twos-starter"):
            if (candidate / "soccer_twos_project" / "notebook_tools.py").exists():
                if str(candidate) not in sys.path:
                    sys.path.insert(0, str(candidate))
                return candidate
    raise FileNotFoundError("Could not find the soccer-twos-starter project root.")

_add_project_root_to_path()

import importlib
import soccer_twos_project.notebook_tools as notebook_tools
importlib.reload(notebook_tools)
from soccer_twos_project.notebook_tools import *

ctx = setup_project()
show_hardware()

## 1. Configuration Knobs

Start with the small run first. Increase `FULL_TIMESTEPS` later only after the debug run works.

In [ ]:
PROFILE_NAME = "auto"  # auto, laptop_cuda, laptop_mps, free_gpu, pro_gpu, cpu_debug
SMALL_TIMESTEPS = 25_000
MEDIUM_TIMESTEPS = 250_000
FULL_TIMESTEPS = None  # None means use the selected profile default

from soccer_twos_project.config import profile_dict, select_profile
profile = select_profile(PROFILE_NAME)
print_json(profile_dict(profile))

## 2. What Files Matter?

This prints the project files you will interact with most. The old example scripts remain available, but the notebook uses the organized helper/runner files.

In [ ]:
important_files = [
    "soccer_twos_project/training.py",
    "soccer_twos_project/envs.py",
    "configs/curriculum.yaml",
    "soccer_twos_project/plotting.py",
    "soccer_twos_project/exporting.py",
    "soccer_twos_project/evaluation.py",
    "soccer_twos_project/imitation.py",
    "soccer_twos_project/notebook_tools.py",
]
for path in important_files:
    print(path, "exists=", (ctx.root / path).exists())

## 3. TensorBoard Monitoring

Ray writes TensorBoard event files under `artifacts/cs8803_soccer_twos/checkpoints`. Run this before training so the plots update during training.

In [ ]:
print("TensorBoard logdir:", ctx.dirs["checkpoints"])
%reload_ext tensorboard
%tensorboard --logdir ./artifacts/cs8803_soccer_twos/checkpoints --reload_interval 10

### Optional: TensorBoard In Browser

If you prefer a browser tab instead of inline notebook TensorBoard, run this and open `http://localhost:6006`. Stop it later with `stop_process(tensorboard_proc)`.

In [ ]:
# tensorboard_proc = launch_tensorboard(ctx, port=6006)

## 4. Understand The Environment

This tiny rollout prints observation/action/reward/debug information. It is not training. It is just to understand what the game loop looks like.

In [ ]:
run_random_debug_episode(max_steps=10)

## 5. Smoke Training Run

This verifies that Ray, Soccer-Twos, checkpointing, TensorBoard, and metadata writing all work. Run this before full training.

In [ ]:
smoke_checkpoint = run_training(
    ctx,
    stage="ppo_baseline",
    profile_name=PROFILE_NAME,
    timesteps=SMALL_TIMESTEPS,
    smoke=True,
    verbose=1,
)
smoke_checkpoint

### Smoke Run Progress

Use these cells repeatedly while/after training. TensorBoard gives the live plot; `show_progress` gives the latest table from Ray's `progress.csv`.

In [ ]:
show_progress(ctx, "ppo_baseline", rows=10)

In [ ]:
plot_progress(ctx, "ppo_baseline")

## 6. Full Agent 1: PPO Baseline

This is the main baseline policy. It trains in `team_vs_policy`, `single_player=True`, `flatten_branched=True`.

In [ ]:
ppo_baseline_checkpoint = run_training(
    ctx,
    stage="ppo_baseline",
    profile_name=PROFILE_NAME,
    timesteps=FULL_TIMESTEPS,
    verbose=1,
)
ppo_baseline_checkpoint

## 7. Full Agent 2: PPO Reward Shaping

This uses the same PPO setup, but `utils.RewardShapingWrapper` adds small training-only bonuses for moving toward the ball and moving the ball toward goal. The submitted agent does not depend on this wrapper.

In [ ]:
ppo_shaped_checkpoint = run_training(
    ctx,
    stage="ppo_shaped",
    profile_name=PROFILE_NAME,
    timesteps=FULL_TIMESTEPS,
    verbose=1,
)
ppo_shaped_checkpoint

## 8. Full Agent 3: PPO Curriculum

This uses `configs/curriculum.yaml` to start with easy ball/player positions and then progress to harder tasks when reward crosses the threshold.

In [ ]:
ppo_curriculum_checkpoint = run_training(
    ctx,
    stage="ppo_curriculum",
    profile_name=PROFILE_NAME,
    timesteps=FULL_TIMESTEPS,
    verbose=1,
)
ppo_curriculum_checkpoint

## 9. Optional Medium Run For Debugging

If full runs are too slow, use this cell to produce report-quality-ish curves without waiting for the full profile default.

In [ ]:
# medium_checkpoint = run_training(ctx, "ppo_baseline", PROFILE_NAME, timesteps=MEDIUM_TIMESTEPS, verbose=1)
# medium_checkpoint

## 10. Compare Training Curves

This generates saved PNG plots for each run and one overlaid comparison plot.

In [ ]:
from types import SimpleNamespace
from soccer_twos_project.plotting import plot_results

plot_results(SimpleNamespace(
    artifact_root=str(ctx.artifact_root),
    ray_results=str(ctx.dirs["checkpoints"]),
    output_dir=str(ctx.dirs["plots"]),
    filter=None,
))

## 11. Export Submission Agents

Export converts RLlib checkpoints into lightweight `AgentInterface` folders containing `agent.py`, `model.py`, `checkpoint.pth`, metadata, README, requirements, and zip files.

In [ ]:
from soccer_twos_project.exporting import export_checkpoint

AUTHOR = "Your Name"
EMAIL = "your.email@gatech.edu"

def export_agent(stage, agent_name, description):
    export_checkpoint(SimpleNamespace(
        checkpoint=best_checkpoint(ctx, stage),
        stage=stage,
        policy_id="default_policy",
        profile="cpu_debug",
        artifact_root=str(ctx.artifact_root),
        output_dir=None,
        agent_name=agent_name,
        author=AUTHOR,
        email=EMAIL,
        description=description,
        no_zip=False,
        clean=True,
    ))

export_agent("ppo_baseline", "soccer_ppo_baseline", "PPO baseline trained in team_vs_policy single-player mode.")
export_agent("ppo_shaped", "soccer_ppo_shaped", "PPO policy trained with distance-based reward shaping.")
export_agent("ppo_curriculum", "soccer_ppo_curriculum", "PPO policy trained with curriculum initialization.")

## 12. Optional Imitation Learning Agent

This collects expert `(observation, action)` pairs and trains a small behavior-cloning classifier. Use `soccer_ppo_curriculum` as the expert after export, or use `ceia_baseline_agent` if you downloaded it.

In [ ]:
from soccer_twos_project.imitation import collect_dataset, train_bc

EXPERT_MODULE = "soccer_ppo_curriculum"
DATASET_PATH = ctx.dirs["datasets"] / "bc_expert_dataset.npz"

collect_dataset(SimpleNamespace(
    expert_module=EXPERT_MODULE,
    samples=50_000,
    output=str(DATASET_PATH),
    base_port=None,
    artifact_root=str(ctx.artifact_root),
))

train_bc(SimpleNamespace(
    dataset=str(DATASET_PATH),
    agent_name="soccer_bc_imitation",
    author=AUTHOR,
    email=EMAIL,
    description="Behavior cloning agent trained from expert Soccer-Twos rollouts.",
    hidden_layers="256,256",
    epochs=20,
    batch_size=256,
    lr=1e-3,
    val_fraction=0.1,
    seed=0,
    output_dir=None,
    artifact_root=str(ctx.artifact_root),
    no_zip=False,
))

## 13. Evaluate Agents

Start with 5 episodes for a quick check. Use 100 episodes for final report metrics.

In [ ]:
from soccer_twos_project.evaluation import evaluate, safe_label, write_outputs

def evaluate_pair(agent1, agent2, episodes=5):
    rows, summary = evaluate(agent1, agent2, episodes=episodes, base_port=None)
    write_outputs(rows, summary, ctx.dirs["evals"], safe_label(agent1, agent2))
    print_json(summary)
    return summary

evaluate_pair("soccer_ppo_baseline", "soccer_ppo_curriculum", episodes=5)

## 14. Final Report Evaluations

Uncomment these once the exported packages and optional `ceia_baseline_agent` are available.

In [ ]:
# evaluate_pair("soccer_ppo_baseline", "ceia_baseline_agent", episodes=100)
# evaluate_pair("soccer_ppo_shaped", "ceia_baseline_agent", episodes=100)
# evaluate_pair("soccer_ppo_curriculum", "ceia_baseline_agent", episodes=100)
# evaluate_pair("soccer_bc_imitation", "ceia_baseline_agent", episodes=100)

## 15. Artifact Checklist

At the end, these folders should contain your run outputs, plots, exported packages, zips, and evaluation CSV/JSON files.

In [ ]:
for name, path in ctx.dirs.items():
    print(f"{name:12s}", path)
    for item in sorted(path.glob("*"))[:10]:
        print("  -", item.name)